## Instalación de dependencias
*Ejecuta esta celda solo si faltan paquetes. Reinicia el kernel después.*

In [1]:
import subprocess, sys

paquetes = ['xgboost', 'imbalanced-learn', 'scikit-learn', 'plotly', 'pandas', 'numpy', 'scipy', 'joblib']

for pkg in paquetes:
    nombre_import = pkg.replace('-', '_').split('[')[0]
    try:
        __import__(nombre_import)
        print(f'OK  {pkg}')
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'OK  {pkg} instalado')


OK  xgboost
Instalando imbalanced-learn...


OK  imbalanced-learn instalado
Instalando scikit-learn...


OK  scikit-learn instalado
OK  plotly
OK  pandas
OK  numpy
OK  scipy
OK  joblib


# Fase 5 — Evaluación del Modelo
## CRISP-DM · Universidad de los Llanos · Cohortes 2017-2 y 2018-1 · Ingeniería de Sistemas

Este notebook verifica que el modelo cumple los objetivos del negocio, analiza sus errores
y produce la decisión de aprobación para despliegue.

**Secciones:** Setup · Criterios de negocio · Matrices de confusión ·
Curvas ROC/PR · Validación cruzada por fold · Análisis de errores ·
Supuestos · Decisión de aprobación

## 1. Setup y carga de artefactos

In [2]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, joblib, json, os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve,
    average_precision_score, f1_score, recall_score,
    precision_score, matthews_corrcoef, accuracy_score, roc_auc_score
)
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from scipy import stats

SEED=42; UMBRAL_RB=0.29; UMBRAL_GR=0.50
SRC = 'src' if os.path.exists('src') else '../src'

df   = pd.read_csv(os.path.join(SRC, 'df_master_limpio.csv'))
FEAT = joblib.load(os.path.join(SRC, 'feature_names.pkl'))
with open(os.path.join(SRC, 'train_test_split.json')) as f:
    split = json.load(f)

X=df[FEAT].values
CV=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
idx_tr=split['idx_train']; idx_te=split['idx_test']

print(f'Dataset: {df.shape[0]} estudiantes | {len(FEAT)} features')
print(f'Train: {len(idx_tr)} | Test: {len(idx_te)}')
print(f'rendimiento_bajo: {df["rendimiento_bajo"].sum()} positivos / {(df["rendimiento_bajo"]==0).sum()} negativos')
print(f'graduado:         {df["graduado"].sum()} positivos / {(df["graduado"]==0).sum()} negativos')


Dataset: 94 estudiantes | 18 features
Train: 75 | Test: 19
rendimiento_bajo: 37 positivos / 57 negativos
graduado:         35 positivos / 59 negativos


In [3]:
def get_cv_probs(target, umbral):
    y = df[target].values
    X_tr, y_tr = X[idx_tr], y[idx_tr]
    counts = np.bincount(y_tr)
    if counts.min() / counts.max() < 0.4 and counts.min() >= 3:
        sm = SMOTE(random_state=SEED, k_neighbors=min(3, counts.min()-1))
        X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
    if target == 'rendimiento_bajo':
        m = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=2, random_state=SEED)
    else:
        m = XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            use_label_encoder=False, eval_metric='logloss',
            verbosity=0, random_state=SEED
        )
    m.fit(X_tr, y_tr)
    prob_cv = cross_val_predict(m, X, y, cv=CV, method='predict_proba')[:, 1]
    return y, prob_cv, (prob_cv >= umbral).astype(int), m

y_rb, prob_rb, yp_rb, modelo_rb = get_cv_probs('rendimiento_bajo', UMBRAL_RB)
y_gr, prob_gr, yp_gr, modelo_gr = get_cv_probs('graduado',         UMBRAL_GR)
print('Modelos entrenados y probabilidades CV-5 obtenidas.')


Modelos entrenados y probabilidades CV-5 obtenidas.


## 2. Evaluación contra criterios de negocio

El objetivo es identificar estudiantes en riesgo para **intervención temprana**.
Criterios mínimos: F1-macro ≥ 0.65 y AUC ≥ 0.70.

In [4]:
rows = []
for target, y, prob, yp, u in [
    ('rendimiento_bajo', y_rb, prob_rb, yp_rb, UMBRAL_RB),
    ('graduado',         y_gr, prob_gr, yp_gr, UMBRAL_GR)]:
    rows.append({
        'Target': target, 'Umbral': u,
        'Recall+':  round(recall_score(y, yp, zero_division=0), 3),
        'Prec+':    round(precision_score(y, yp, zero_division=0), 3),
        'F1-mac':   round(f1_score(y, yp, average='macro'), 3),
        'AUC':      round(roc_auc_score(y, prob), 3),
        'AvgPrec':  round(average_precision_score(y, prob), 3),
        'MCC':      round(matthews_corrcoef(y, yp), 3),
        'F1>=0.65': 'OK' if f1_score(y, yp, average='macro') >= 0.65 else 'FALLA',
        'AUC>=0.70':'OK' if roc_auc_score(y, prob) >= 0.70 else 'FALLA',
    })
df_neg = pd.DataFrame(rows)
print(df_neg.to_string(index=False))
todos_ok = all(df_neg['F1>=0.65'] == 'OK') and all(df_neg['AUC>=0.70'] == 'OK')
print('\nCriterios de negocio:', 'TODOS CUMPLIDOS' if todos_ok else 'INCUMPLIDOS')


          Target  Umbral  Recall+  Prec+  F1-mac   AUC  AvgPrec   MCC F1>=0.65 AUC>=0.70
rendimiento_bajo    0.29    0.757  0.737   0.789 0.827    0.821 0.579       OK        OK
        graduado    0.50    0.571  0.690   0.716 0.765    0.665 0.438       OK        OK

Criterios de negocio: TODOS CUMPLIDOS


## 3. Matrices de confusión (CV-5, n=94)

Cada celda muestra el número de estudiantes clasificados en esa categoría
sobre los 94 del dataset completo mediante validación cruzada estratificada.

In [5]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=[
        f'Rendimiento Bajo (umbral {UMBRAL_RB})',
        f'Graduacion (umbral {UMBRAL_GR})'
    ])

for col, (y, yp, labs) in enumerate([
    (y_rb, yp_rb, ['Normal', 'Riesgo']),
    (y_gr, yp_gr, ['No Graduado', 'Graduado'])], 1):
    cm = confusion_matrix(y, yp)
    fig.add_trace(
        go.Heatmap(z=cm, colorscale='Blues', showscale=False,
                   x=['Pred. ' + l for l in labs],
                   y=['Real ' + l for l in labs]),
        row=1, col=col)
    for i in range(2):
        for j in range(2):
            fig.add_annotation(
                x=j, y=i, text=str(cm[i, j]), showarrow=False,
                font=dict(size=18, color='white' if cm[i,j] > cm.max()/2 else 'black'),
                xref=f'x{col}', yref=f'y{col}')

fig.update_layout(
    title_text='Matrices de Confusion - CV-5 (n=94)',
    title_font_size=13, height=400, width=780)
fig.show()


## 4. Curvas ROC y Precisión-Recall (CV-5)

El **punto rojo** marca el umbral de operación seleccionado.
La línea punteada en PR representa la tasa de positivos (clasificador base sin información).

In [6]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=[
        'ROC - Rend. Bajo', 'ROC - Graduacion',
        'Prec-Recall - Rend. Bajo', 'Prec-Recall - Graduacion'
    ])

for col, (y, prob, yp, u) in enumerate([
    (y_rb, prob_rb, yp_rb, UMBRAL_RB),
    (y_gr, prob_gr, yp_gr, UMBRAL_GR)], 1):
    # ROC
    fpr, tpr, _ = roc_curve(y, prob)
    auc_v = auc(fpr, tpr)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
        name=f'AUC={auc_v:.3f}', line=dict(width=2)), row=1, col=col)
    fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', showlegend=False,
        line=dict(dash='dash', color='gray', width=1)), row=1, col=col)
    fp_r = ((yp==1) & (y==0)).sum() / (y==0).sum()
    tp_r = recall_score(y, yp)
    fig.add_trace(go.Scatter(x=[fp_r], y=[tp_r], mode='markers',
        name=f'Umbral {u}', showlegend=(col==1),
        marker=dict(color='red', size=10)), row=1, col=col)
    # PR
    prec_c, rec_c, _ = precision_recall_curve(y, prob)
    ap = average_precision_score(y, prob)
    fig.add_trace(go.Scatter(x=rec_c, y=prec_c, mode='lines', showlegend=False,
        name=f'AvgPrec={ap:.3f}', line=dict(width=2)), row=2, col=col)
    fig.add_hline(y=y.mean(), line_dash='dot', line_color='gray',
        annotation_text=f'Base {y.mean():.2f}', row=2, col=col)
    fig.add_trace(go.Scatter(
        x=[recall_score(y, yp)], y=[precision_score(y, yp, zero_division=0)],
        mode='markers', showlegend=False,
        marker=dict(color='red', size=10)), row=2, col=col)

fig.update_xaxes(title_text='FPR', row=1)
fig.update_yaxes(title_text='TPR', row=1)
fig.update_xaxes(title_text='Recall', row=2)
fig.update_yaxes(title_text='Precision', row=2)
fig.update_layout(height=680, width=880,
    title_text='Curvas ROC y Precision-Recall - CV-5',
    legend=dict(orientation='h', y=-0.08))
fig.show()


## 5. Validación cruzada — detalle por fold

Permite evaluar la **estabilidad** del modelo. Alta varianza entre folds indica
sensibilidad al subconjunto de entrenamiento — esperable con n=94.

In [7]:
def cv_folds(target, umbral):
    y = df[target].values
    X_tr2, y_tr2 = X[idx_tr], y[idx_tr]
    counts = np.bincount(y_tr2)
    if counts.min() / counts.max() < 0.4 and counts.min() >= 3:
        sm = SMOTE(random_state=SEED, k_neighbors=min(3, counts.min()-1))
        X_tr2, y_tr2 = sm.fit_resample(X_tr2, y_tr2)
        if target == 'rendimiento_bajo':
            m = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=2, random_state=SEED)
        else:
            m = XGBClassifier(
                n_estimators=100, max_depth=3, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8,
                use_label_encoder=False, eval_metric='logloss',
                verbosity=0, random_state=SEED
            )
    rows = []
    for fold, (tr_i, te_i) in enumerate(CV.split(X, y), 1):
        m.fit(X[tr_i], y[tr_i])
        prob_i = m.predict_proba(X[te_i])[:, 1]
        yp_i   = (prob_i >= umbral).astype(int)
        rows.append({'Fold': fold, 'N_test': len(te_i), 'N_pos': int(y[te_i].sum()),
            'Recall+': round(recall_score(y[te_i], yp_i, zero_division=0), 3),
            'F1-mac':  round(f1_score(y[te_i], yp_i, average='macro', zero_division=0), 3),
            'MCC':     round(matthews_corrcoef(y[te_i], yp_i), 3)})
    return pd.DataFrame(rows)

df_cv_rb = cv_folds('rendimiento_bajo', UMBRAL_RB)
df_cv_gr = cv_folds('graduado',         UMBRAL_GR)

for dfc, nombre in [(df_cv_rb, 'Rendimiento Bajo'), (df_cv_gr, 'Graduado')]:
    print(f'=== {nombre} ===')
    print(dfc.to_string(index=False))
    print(f'  Recall+ : {dfc["Recall+"].mean():.3f} +/- {dfc["Recall+"].std():.3f}')
    print(f'  F1-mac  : {dfc["F1-mac"].mean():.3f} +/- {dfc["F1-mac"].std():.3f}')
    print(f'  MCC     : {dfc["MCC"].mean():.3f} +/- {dfc["MCC"].std():.3f}\n')


=== Rendimiento Bajo ===
 Fold  N_test  N_pos  Recall+  F1-mac   MCC
    1      19      7    0.571   0.756 0.535
    2      19      7    1.000   0.945 0.896
    3      19      8    0.625   0.774 0.567
    4      19      8    0.750   0.734 0.472
    5      18      7    0.857   0.721 0.484
  Recall+ : 0.761 +/- 0.174
  F1-mac  : 0.786 +/- 0.091
  MCC     : 0.591 +/- 0.175

=== Graduado ===
 Fold  N_test  N_pos  Recall+  F1-mac   MCC
    1      19      7    0.429   0.548 0.095
    2      19      7    0.571   0.756 0.535
    3      19      7    0.714   0.774 0.548
    4      19      7    0.714   0.825 0.655
    5      18      7    0.429   0.673 0.396
  Recall+ : 0.571 +/- 0.143
  F1-mac  : 0.715 +/- 0.108
  MCC     : 0.446 +/- 0.217



In [8]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Rendimiento Bajo (u=0.29)', 'Graduacion (u=0.50)'])

metrics = ['Recall+', 'F1-mac', 'MCC']
colors  = ['#1f77b4', '#ff7f0e', '#2ca02c']

for col, dfc in enumerate([df_cv_rb, df_cv_gr], 1):
    for metric, color in zip(metrics, colors):
        media = dfc[metric].mean()
        fig.add_trace(go.Bar(
            x=[f'Fold {i}' for i in range(1, 6)],
            y=dfc[metric].tolist(),
            name=f'{metric} (mu={media:.3f})',
            marker_color=color,
            showlegend=(col == 1)
        ), row=1, col=col)

fig.update_layout(
    barmode='group', height=420, width=880,
    title_text='Metricas por Fold - StratifiedKFold k=5',
    legend=dict(orientation='h', y=-0.2))
fig.show()


## 6. Análisis de errores

Se compara el perfil de los **falsos negativos** (en riesgo no detectados) con los
**falsos positivos** (falsas alarmas) y la media general.

Los falsos negativos son el error más costoso en contexto educativo: el estudiante no recibe intervención.

In [9]:
fn_mask = (y_rb == 1) & (yp_rb == 0)
fp_mask = (y_rb == 0) & (yp_rb == 1)
tp_mask = (y_rb == 1) & (yp_rb == 1)

print(f'Falsos negativos (no detectados):  {fn_mask.sum()}')
print(f'Falsos positivos (falsas alarmas): {fp_mask.sum()}')
print(f'Verdaderos positivos (correctos):  {tp_mask.sum()}')

feats = ['prom_sem1', 'icfes_total', 'nivel_edu_max_padres', 'estrato', 'log_ingresos']
labs  = ['prom_sem1', 'ICFES total', 'Edu max padres', 'Estrato', 'Log ingresos']

perfiles = pd.DataFrame({
    'General':      df[feats].mean(),
    'FN (perdidos)':df[fn_mask][feats].mean() if fn_mask.sum() > 0 else pd.Series([np.nan]*len(feats), index=feats),
    'FP (alarmas)': df[fp_mask][feats].mean() if fp_mask.sum() > 0 else pd.Series([np.nan]*len(feats), index=feats),
    'TP (aciertos)':df[tp_mask][feats].mean() if tp_mask.sum() > 0 else pd.Series([np.nan]*len(feats), index=feats),
}).round(3)
perfiles.index = labs
print()
print(perfiles.to_string())


Falsos negativos (no detectados):  9
Falsos positivos (falsas alarmas): 10
Verdaderos positivos (correctos):  28

                General  FN (perdidos)  FP (alarmas)  TP (aciertos)
prom_sem1         3.276          3.678         3.410          2.304
ICFES total     317.862        319.111       332.700        317.286
Edu max padres    6.521          6.778         5.300          6.250
Estrato           1.947          2.333         2.200          1.821
Log ingresos     16.290         16.414        16.463         16.302


In [10]:
fig = go.Figure()
grupos  = ['General', 'FN (perdidos)', 'FP (alarmas)', 'TP (aciertos)']
colores = ['#7f7f7f', '#d62728', '#ff7f0e', '#2ca02c']

for grupo, color in zip(grupos, colores):
    fig.add_trace(go.Bar(
        name=grupo, x=labs,
        y=perfiles[grupo].tolist(),
        marker_color=color
    ))

fig.update_layout(
    barmode='group', height=430, width=820,
    title='Perfil de errores vs. aciertos - Rendimiento Bajo (CV-5)',
    xaxis_title='Feature', yaxis_title='Valor promedio',
    legend=dict(orientation='h', y=-0.25))
fig.show()


**Interpretación:**
- **Falsos negativos** tienen `prom_sem1` más alto que la media: parecen 'normales' al ingreso pero se deterioran después.
- **Falsos positivos** tienen primer semestre débil pero logran recuperarse.
- **Límite del modelo:** no puede anticipar trayectorias que cambian de dirección tras el primer semestre.

## 7. Revisión de supuestos

XGBoost (ensamble de árboles) **no asume linealidad, normalidad ni homocedasticidad**.
Se verifican los supuestos relevantes para este tipo de modelo y dataset.

In [11]:
supuestos = [
    ('Independencia de observaciones', 'OK',    'Cada fila es un estudiante unico'),
    ('Ausencia de data leakage',       'OK',    'PROMEDIO_CARRERA excluido; SMOTE solo en train'),
    ('Representatividad del train',    'AVISO', 'Solo 2 cohortes de un programa'),
    ('Estabilidad temporal',           'AVISO', 'cohorte_encoded requiere reentrenamiento futuro'),
    ('Suficiencia muestral',           'AVISO', 'n=94 pequeno; mitigado con CV-5'),
    ('Desbalance de clases',           'OK',    'SMOTE aplicado cuando ratio < 0.60, solo en train'),
    ('Multicolinealidad',              'N/A',   'No aplica a XGBoost'),
    ('Normalidad de residuos',         'N/A',   'No aplica a XGBoost'),
]

print(f'{"Supuesto":<40} {"Estado":<8} Detalle')
print('-' * 88)
for s, e, d in supuestos:
    icon = 'OK' if e == 'OK' else ('!!' if e == 'AVISO' else '--')
    print(f'  [{icon}] {s:<38} {d}')


Supuesto                                 Estado   Detalle
----------------------------------------------------------------------------------------
  [OK] Independencia de observaciones         Cada fila es un estudiante unico
  [OK] Ausencia de data leakage               PROMEDIO_CARRERA excluido; SMOTE solo en train
  [!!] Representatividad del train            Solo 2 cohortes de un programa
  [!!] Estabilidad temporal                   cohorte_encoded requiere reentrenamiento futuro
  [!!] Suficiencia muestral                   n=94 pequeno; mitigado con CV-5
  [OK] Desbalance de clases                   SMOTE aplicado cuando ratio < 0.60, solo en train
  [--] Multicolinealidad                      No aplica a XGBoost
  [--] Normalidad de residuos                 No aplica a XGBoost


## 8. Decisión de aprobación

In [12]:
criterios = [
    ('Recall+ rendimiento_bajo >= 0.75',
     recall_score(y_rb, yp_rb) >= 0.75,
     recall_score(y_rb, yp_rb)),
    ('F1-macro rendimiento_bajo >= 0.65',
     f1_score(y_rb, yp_rb, average='macro') >= 0.65,
     f1_score(y_rb, yp_rb, average='macro')),
    ('AUC rendimiento_bajo >= 0.70',
     roc_auc_score(y_rb, prob_rb) >= 0.70,
     roc_auc_score(y_rb, prob_rb)),
    ('F1-macro graduado >= 0.65',
     f1_score(y_gr, yp_gr, average='macro') >= 0.65,
     f1_score(y_gr, yp_gr, average='macro')),
    ('AUC graduado >= 0.70',
     roc_auc_score(y_gr, prob_gr) >= 0.70,
     roc_auc_score(y_gr, prob_gr)),
    ('Modelos materias criticas >= 3',
     len(joblib.load(os.path.join(SRC, 'modelos_materias.pkl'))) >= 3,
     len(joblib.load(os.path.join(SRC, 'modelos_materias.pkl')))),
]

todos_ok = all(c[1] for c in criterios)

print('CRITERIOS DE APROBACION')
print('=' * 55)
for nombre, cumple, valor in criterios:
    v_str = str(round(float(valor), 3)) if isinstance(valor, (int, float, np.floating)) else str(valor)
    estado = 'OK   ' if cumple else 'FALLA'
    print(f'  [{estado}]  {nombre}: {v_str}')

print()
print('=' * 55)
decision = 'SI - CON CONDICIONES' if todos_ok else 'NO - REQUIERE REVISION'
print(f'  DECISION: {decision}')
print('=' * 55)

if todos_ok:
    print('\nCondiciones para el despliegue:')
    print('  1. Usar umbral 0.29 para rendimiento_bajo (maximiza Recall+)')
    print('  2. Comunicar limitaciones: n=94, 2 cohortes, 84% masculino')
    print('  3. Reentrenar al incorporar nuevas cohortes')
    print('  4. Usar como herramienta de alerta temprana, no como decision definitiva')


CRITERIOS DE APROBACION
  [OK   ]  Recall+ rendimiento_bajo >= 0.75: 0.757
  [OK   ]  F1-macro rendimiento_bajo >= 0.65: 0.789
  [OK   ]  AUC rendimiento_bajo >= 0.70: 0.827
  [OK   ]  F1-macro graduado >= 0.65: 0.716
  [OK   ]  AUC graduado >= 0.70: 0.765
  [OK   ]  Modelos materias criticas >= 3: 4.0

  DECISION: SI - CON CONDICIONES

Condiciones para el despliegue:
  1. Usar umbral 0.29 para rendimiento_bajo (maximiza Recall+)
  2. Comunicar limitaciones: n=94, 2 cohortes, 84% masculino
  3. Reentrenar al incorporar nuevas cohortes
  4. Usar como herramienta de alerta temprana, no como decision definitiva
